# Build filing-level financial tables

This notebook maps the neutral SEC Company Facts data to:

- `data/curated/filing_financials_long.parquet`: one selected mapped fact per filing-variable, with provenance.
- `data/curated/filing_financials_wide.parquet`: one row per original 10-K filing.

The mapping remains explicit in `config/financial_concept_map.csv`.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FACTS_GLOB = (PROJECT_ROOT / "data/base/financial_facts/*.parquet").resolve().as_posix()
FILING_INDEX_PATH = (PROJECT_ROOT / "data/curated/filing_index.parquet").resolve().as_posix()
CONFIG_DIR = PROJECT_ROOT / "config"
CURATED_DIR = PROJECT_ROOT / "data/curated"

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
CURATED_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE VIEW financial_facts AS
SELECT * FROM read_parquet('{FACTS_GLOB}')
""")

con.execute(f"""
CREATE OR REPLACE VIEW filing_index AS
SELECT * FROM read_parquet('{FILING_INDEX_PATH}')
""")

con.execute("""
CREATE OR REPLACE VIEW annual_filings AS
SELECT
    cik,
    accession_number,
    filing_date,
    report_date,
    acceptance_datetime,
    form,
    is_xbrl,
    is_inline_xbrl,
    primary_document,
    filing_url
FROM filing_index
WHERE form = '10-K'
  AND report_date IS NOT NULL
""")

In [2]:
con.sql("""
SELECT
    COUNT(*) AS filings,
    COUNT(DISTINCT cik) AS filers,
    MIN(filing_date) AS first_filing_date,
    MAX(filing_date) AS last_filing_date
FROM annual_filings
""").df()

,filings,filers,first_filing_date,last_filing_date
0,238680,39422,1993-11-29,2026-07-10


## Concept mapping

`priority = 1` is preferred. Higher numbers are fallbacks; they are not added together.

This is a first version to inspect and revise.

In [3]:
concept_map = pd.DataFrame(
    [
        ("assets", 1, "us-gaap", "Assets", "USD", "instant"),
        ("liabilities", 1, "us-gaap", "Liabilities", "USD", "instant"),
        ("equity", 1, "us-gaap", "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest", "USD", "instant"),
        ("equity", 2, "us-gaap", "StockholdersEquity", "USD", "instant"),
        ("current_assets", 1, "us-gaap", "AssetsCurrent", "USD", "instant"),
        ("current_liabilities", 1, "us-gaap", "LiabilitiesCurrent", "USD", "instant"),
        ("cash", 1, "us-gaap", "CashAndCashEquivalentsAtCarryingValue", "USD", "instant"),
        ("cash", 2, "us-gaap", "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents", "USD", "instant"),
        ("revenue", 1, "us-gaap", "RevenueFromContractWithCustomerExcludingAssessedTax", "USD", "duration"),
        ("revenue", 2, "us-gaap", "Revenues", "USD", "duration"),
        ("revenue", 3, "us-gaap", "SalesRevenueNet", "USD", "duration"),
        ("net_income", 1, "us-gaap", "NetIncomeLoss", "USD", "duration"),
        ("net_income", 2, "us-gaap", "ProfitLoss", "USD", "duration"),
        ("operating_income", 1, "us-gaap", "OperatingIncomeLoss", "USD", "duration"),
        ("operating_cash_flow", 1, "us-gaap", "NetCashProvidedByUsedInOperatingActivities", "USD", "duration"),
        ("interest_expense", 1, "us-gaap", "InterestExpenseNonOperating", "USD", "duration"),
        ("interest_expense", 2, "us-gaap", "InterestExpense", "USD", "duration"),
    ],
    columns=["variable", "priority", "taxonomy", "concept", "unit", "period_type"],
)

concept_map.to_csv(CONFIG_DIR / "financial_concept_map.csv", index=False)
con.register("concept_map", concept_map)
concept_map

,variable,priority,taxonomy,concept,unit,period_type
0,assets,1,us-gaap,Assets,USD,instant
1,liabilities,1,us-gaap,Liabilities,USD,instant
2,equity,1,us-gaap,StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest,USD,instant
3,equity,2,us-gaap,StockholdersEquity,USD,instant
4,current_assets,1,us-gaap,AssetsCurrent,USD,instant
5,current_liabilities,1,us-gaap,LiabilitiesCurrent,USD,instant
6,cash,1,us-gaap,CashAndCashEquivalentsAtCarryingValue,USD,instant
7,cash,2,us-gaap,CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents,USD,instant
8,revenue,1,us-gaap,RevenueFromContractWithCustomerExcludingAssessedTax,USD,duration
9,revenue,2,us-gaap,Revenues,USD,duration


## Raw coverage of mapped concepts

In [4]:
con.sql("""
SELECT
    m.variable,
    m.priority,
    m.concept,
    m.period_type,
    COUNT(*) AS observations,
    COUNT(DISTINCT (f.cik, f.accession_number)) AS filings,
    COUNT(DISTINCT f.cik) AS filers
FROM financial_facts f
JOIN concept_map m
  ON f.taxonomy = m.taxonomy
 AND f.concept = m.concept
 AND f.unit = m.unit
JOIN annual_filings a
  ON f.cik = a.cik
 AND f.accession_number = a.accession_number
WHERE f.value_numeric IS NOT NULL
GROUP BY m.variable, m.priority, m.concept, m.period_type
ORDER BY m.variable, m.priority
""").df()

,variable,priority,concept,period_type,observations,filings,filers
0,assets,1,Assets,instant,196556,92688,13851
1,cash,1,CashAndCashEquivalentsAtCarryingValue,instant,244617,81210,12644
2,cash,2,CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents,instant,108480,30540,7086
3,current_assets,1,AssetsCurrent,instant,145206,72585,11213
4,current_liabilities,1,LiabilitiesCurrent,instant,145305,72381,11167
5,equity,1,StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest,instant,107393,30014,5146
6,equity,2,StockholdersEquity,instant,267878,85708,13156
7,interest_expense,2,InterestExpense,duration,172808,53713,9301
8,liabilities,1,Liabilities,instant,151433,75383,12169
9,net_income,1,NetIncomeLoss,duration,444147,86640,13492


## Period-matched candidates

Initial rules:

- Instant: `end_date = report_date`
- Duration: `end_date = report_date` and 250–450 days

The duration range is deliberately broad for inspection.

In [5]:
con.execute("""
CREATE OR REPLACE VIEW mapped_candidates AS
SELECT
    f.cik,
    f.accession_number,
    a.filing_date,
    a.report_date,
    a.acceptance_datetime,
    a.form,

    m.variable,
    m.priority,
    m.period_type,

    f.taxonomy AS source_taxonomy,
    f.concept AS source_concept,
    f.label AS source_label,
    f.unit AS source_unit,
    f.value_numeric AS value,
    f.value_text,
    f.start_date AS source_start_date,
    f.end_date AS source_end_date,
    f.filed_date AS source_filed_date,
    f.fiscal_year AS source_fiscal_year,
    f.fiscal_period AS source_fiscal_period,
    f.frame AS source_frame,
    f.source_member,
    f.source_observation_index,

    CASE
        WHEN f.start_date IS NULL THEN NULL
        ELSE DATE_DIFF('day', f.start_date, f.end_date)
    END AS duration_days,

    CASE
        WHEN m.period_type = 'instant'
             AND f.end_date = a.report_date
        THEN TRUE
        WHEN m.period_type = 'duration'
             AND f.end_date = a.report_date
             AND f.start_date IS NOT NULL
             AND DATE_DIFF('day', f.start_date, f.end_date) BETWEEN 250 AND 450
        THEN TRUE
        ELSE FALSE
    END AS period_matches,

    CASE
        WHEN m.period_type = 'instant' THEN 0
        WHEN f.start_date IS NULL THEN NULL
        ELSE ABS(DATE_DIFF('day', f.start_date, f.end_date) - 365)
    END AS annual_period_distance

FROM financial_facts f
JOIN concept_map m
  ON f.taxonomy = m.taxonomy
 AND f.concept = m.concept
 AND f.unit = m.unit
JOIN annual_filings a
  ON f.cik = a.cik
 AND f.accession_number = a.accession_number
WHERE f.value_numeric IS NOT NULL
""")

In [6]:
con.sql("""
SELECT
    variable,
    period_type,
    COUNT(*) AS candidate_rows,
    SUM(CASE WHEN period_matches THEN 1 ELSE 0 END) AS matched_rows,
    COUNT(DISTINCT CASE WHEN period_matches THEN (cik, accession_number) END) AS matched_filings
FROM mapped_candidates
GROUP BY variable, period_type
ORDER BY variable
""").df()

,variable,period_type,candidate_rows,matched_rows,matched_filings
0,assets,instant,196556,92159.0,92159
1,cash,instant,353097,110503.0,85416
2,current_assets,instant,145206,72031.0,72031
3,current_liabilities,instant,145305,72055.0,72055
4,equity,instant,375271,115055.0,89472
5,interest_expense,duration,172808,52486.0,52452
6,liabilities,instant,151433,75056.0,75056
7,net_income,duration,609752,124585.0,90390
8,operating_cash_flow,duration,207512,78602.0,78448
9,operating_income,duration,298544,70140.0,70071


In [7]:
con.sql("""
SELECT
    variable,
    APPROX_QUANTILE(duration_days, 0.01) AS p01,
    APPROX_QUANTILE(duration_days, 0.05) AS p05,
    APPROX_QUANTILE(duration_days, 0.50) AS median,
    APPROX_QUANTILE(duration_days, 0.95) AS p95,
    APPROX_QUANTILE(duration_days, 0.99) AS p99
FROM mapped_candidates
WHERE period_type = 'duration'
  AND source_end_date = report_date
  AND duration_days IS NOT NULL
GROUP BY variable
ORDER BY variable
""").df()

,variable,p01,p05,median,p95,p99
0,interest_expense,91,91,364,365,2704
1,net_income,90,91,364,365,2214
2,operating_cash_flow,292,363,364,365,2731
3,operating_income,90,91,364,365,2149
4,revenue,90,91,364,365,1406


## Select one fact per filing-variable

Selection order:

1. Lowest mapping priority
2. Annual duration closest to 365 days
3. Stable source ordering

Different tied values are flagged rather than hidden.

In [8]:
con.execute("""
CREATE OR REPLACE VIEW selected_filing_financials_long AS
WITH valid AS (
    SELECT * FROM mapped_candidates WHERE period_matches
),
best_priority AS (
    SELECT cik, accession_number, variable, MIN(priority) AS best_priority
    FROM valid
    GROUP BY cik, accession_number, variable
),
priority_candidates AS (
    SELECT v.*
    FROM valid v
    JOIN best_priority p
      ON v.cik = p.cik
     AND v.accession_number = p.accession_number
     AND v.variable = p.variable
     AND v.priority = p.best_priority
),
best_distance AS (
    SELECT
        cik,
        accession_number,
        variable,
        MIN(COALESCE(annual_period_distance, 999999)) AS best_period_distance
    FROM priority_candidates
    GROUP BY cik, accession_number, variable
),
finalists AS (
    SELECT p.*
    FROM priority_candidates p
    JOIN best_distance d
      ON p.cik = d.cik
     AND p.accession_number = d.accession_number
     AND p.variable = d.variable
     AND COALESCE(p.annual_period_distance, 999999) = d.best_period_distance
),
tie_summary AS (
    SELECT
        cik,
        accession_number,
        variable,
        COUNT(*) AS tied_candidate_count,
        COUNT(DISTINCT value_text) AS tied_distinct_value_count
    FROM finalists
    GROUP BY cik, accession_number, variable
),
ranked AS (
    SELECT
        f.*,
        ROW_NUMBER() OVER (
            PARTITION BY f.cik, f.accession_number, f.variable
            ORDER BY f.source_member, f.source_observation_index
        ) AS selection_rank
    FROM finalists f
)
SELECT
    r.cik,
    r.accession_number,
    r.filing_date,
    r.report_date,
    r.acceptance_datetime,
    r.form,
    r.variable,
    r.value,
    r.source_taxonomy,
    r.source_concept,
    r.source_label,
    r.source_unit,
    r.source_start_date,
    r.source_end_date,
    r.source_filed_date,
    r.source_fiscal_year,
    r.source_fiscal_period,
    r.source_frame,
    r.priority AS mapping_priority,
    r.period_type,
    r.duration_days,
    r.annual_period_distance,
    t.tied_candidate_count,
    t.tied_distinct_value_count,
    CASE
        WHEN t.tied_distinct_value_count > 1 THEN 'selected_ambiguous_value'
        WHEN t.tied_candidate_count > 1 THEN 'selected_duplicate_value'
        ELSE 'selected'
    END AS selection_status
FROM ranked r
JOIN tie_summary t
  ON r.cik = t.cik
 AND r.accession_number = t.accession_number
 AND r.variable = t.variable
WHERE r.selection_rank = 1
""")

In [9]:
con.sql("""
SELECT
    variable,
    COUNT(*) AS selected_filings,
    SUM(CASE WHEN selection_status = 'selected_ambiguous_value' THEN 1 ELSE 0 END) AS ambiguous_value_filings,
    SUM(CASE WHEN selection_status = 'selected_duplicate_value' THEN 1 ELSE 0 END) AS duplicate_value_filings,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM annual_filings), 2) AS annual_filing_coverage_pct
FROM selected_filing_financials_long
GROUP BY variable
ORDER BY variable
""").df()

,variable,selected_filings,ambiguous_value_filings,duplicate_value_filings,annual_filing_coverage_pct
0,assets,92159,0.0,0.0,38.61
1,cash,85416,0.0,0.0,35.79
2,current_assets,72031,0.0,0.0,30.18
3,current_liabilities,72055,0.0,0.0,30.19
4,equity,89472,0.0,0.0,37.49
5,interest_expense,52452,0.0,0.0,21.98
6,liabilities,75056,0.0,0.0,31.45
7,net_income,90390,0.0,0.0,37.87
8,operating_cash_flow,78448,0.0,0.0,32.87
9,operating_income,70071,0.0,0.0,29.36


In [10]:
con.sql("""
SELECT *
FROM selected_filing_financials_long
WHERE selection_status = 'selected_ambiguous_value'
ORDER BY filing_date DESC
LIMIT 100
""").df()

,cik,accession_number,filing_date,report_date,acceptance_datetime,form,variable,value,source_taxonomy,source_concept,source_label,source_unit,source_start_date,source_end_date,source_filed_date,source_fiscal_year,source_fiscal_period,source_frame,mapping_priority,period_type,duration_days,annual_period_distance,tied_candidate_count,tied_distinct_value_count,selection_status


## Write the auditable long table

In [11]:
LONG_OUTPUT = (CURATED_DIR / "filing_financials_long.parquet").resolve().as_posix()

con.execute(f"""
COPY (
    SELECT *
    FROM selected_filing_financials_long
    ORDER BY cik, filing_date, accession_number, variable
)
TO '{LONG_OUTPUT}'
(FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 250000)
""")

print(LONG_OUTPUT)

C:/Users/niels/OneDrive/Dokumenter/sec-distress-model/data/curated/filing_financials_long.parquet


## Pivot to one row per filing

In [12]:
variables = concept_map["variable"].drop_duplicates().tolist()

value_columns = ",\n".join(
    f"MAX(value) FILTER (WHERE variable = '{v}') AS {v}"
    for v in variables
)

ambiguity_columns = ",\n".join(
    (
        f"MAX(CASE WHEN variable = '{v}' "
        "AND selection_status = 'selected_ambiguous_value' "
        f"THEN 1 ELSE 0 END) AS {v}_ambiguous"
    )
    for v in variables
)

wide_query = f"""
SELECT
    a.cik,
    a.accession_number,
    a.filing_date,
    a.report_date,
    a.acceptance_datetime,
    a.form,
    a.is_xbrl,
    a.is_inline_xbrl,
    a.primary_document,
    a.filing_url,

    {value_columns},

    {ambiguity_columns},

    COUNT(s.variable) AS mapped_variables_available

FROM annual_filings a
LEFT JOIN selected_filing_financials_long s
  ON a.cik = s.cik
 AND a.accession_number = s.accession_number

GROUP BY
    a.cik,
    a.accession_number,
    a.filing_date,
    a.report_date,
    a.acceptance_datetime,
    a.form,
    a.is_xbrl,
    a.is_inline_xbrl,
    a.primary_document,
    a.filing_url
"""

con.execute(f"CREATE OR REPLACE VIEW filing_financials_wide AS {wide_query}")

In [13]:
con.sql("""
SELECT
    COUNT(*) AS filings,
    COUNT(DISTINCT cik) AS filers,
    AVG(mapped_variables_available) AS mean_mapped_variables,
    APPROX_QUANTILE(mapped_variables_available, 0.50) AS median_mapped_variables
FROM filing_financials_wide
""").df()

,filings,filers,mean_mapped_variables,median_mapped_variables
0,238680,39422,3.525695,0


In [14]:
WIDE_OUTPUT = (CURATED_DIR / "filing_financials_wide.parquet").resolve().as_posix()

con.execute(f"""
COPY (
    SELECT *
    FROM filing_financials_wide
    ORDER BY cik, filing_date, accession_number
)
TO '{WIDE_OUTPUT}'
(FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 250000)
""")

print(WIDE_OUTPUT)

C:/Users/niels/OneDrive/Dokumenter/sec-distress-model/data/curated/filing_financials_wide.parquet


## Inspect the result

In [15]:
con.sql("""
SELECT *
FROM filing_financials_wide
WHERE mapped_variables_available > 0
ORDER BY filing_date DESC
LIMIT 30
""").df()

,cik,accession_number,filing_date,report_date,acceptance_datetime,form,is_xbrl,is_inline_xbrl,primary_document,filing_url,assets,liabilities,equity,current_assets,current_liabilities,cash,revenue,net_income,operating_income,operating_cash_flow,interest_expense,assets_ambiguous,liabilities_ambiguous,equity_ambiguous,current_assets_ambiguous,current_liabilities_ambiguous,cash_ambiguous,revenue_ambiguous,net_income_ambiguous,operating_income_ambiguous,operating_cash_flow_ambiguous,interest_expense_ambiguous,mapped_variables_available
0,0001673481,0001493152-26-032786,2026-07-10,2025-12-31,2026-07-10 11:27:02+02:00,10-K,1,1,form10-k.htm,https://www.sec.gov/Archives/edgar/data/1673481/000149315226032786/form10-k.htm,5.566022e+07,3.189936e+07,2.376087e+07,1.288029e+07,3.189936e+07,171524.0,5.595900e+05,-20303608.0,-17867883.0,-3.430421e+06,NaN,0,0,0,0,0,0,0,0,0,0,0,10
1,0002020919,0001683168-26-005450,2026-07-10,2026-03-31,2026-07-10 17:17:13+02:00,10-K,1,1,deltasoft_i10k-033126.htm,https://www.sec.gov/Archives/edgar/data/2020919/000168316826005450/deltasoft_i10k-033126.htm,7.333800e+04,3.447000e+04,3.886800e+04,4.547000e+04,3.447000e+04,25470.0,3.791800e+04,-76.0,-13757.0,-2.840100e+04,NaN,0,0,0,0,0,0,0,0,0,0,0,10
2,0000849401,0001437749-26-023316,2026-07-10,2026-03-31,2026-07-10 18:00:58+02:00,10-K,1,1,admt20260331_10k.htm,https://www.sec.gov/Archives/edgar/data/849401/000143774926023316/admt20260331_10k.htm,1.826512e+06,1.255540e+06,5.709720e+05,1.022014e+06,1.046312e+06,255730.0,NaN,-100374.0,-159097.0,-1.240590e+05,NaN,0,0,0,0,0,0,0,0,0,0,0,9
3,0002056016,0001493152-26-032759,2026-07-10,2026-04-30,2026-07-10 12:10:45+02:00,10-K,1,1,form10-k.htm,https://www.sec.gov/Archives/edgar/data/2056016/000149315226032759/form10-k.htm,5.497600e+04,3.974400e+04,1.523200e+04,1.656000e+04,3.767300e+04,14695.0,4.025400e+04,-23622.0,-24536.0,-2.605400e+04,NaN,0,0,0,0,0,0,0,0,0,0,0,10
4,0001912954,0001912954-26-000008,2026-07-09,2025-12-31,2026-07-09 18:38:23+02:00,10-K,1,1,wdfi-20251231_10k.htm,https://www.sec.gov/Archives/edgar/data/1912954/000191295426000008/wdfi-20251231_10k.htm,1.015280e+05,8.700000e+03,9.282800e+04,1.146200e+04,8.700000e+03,6568.0,1.632000e+03,-21464.0,NaN,-1.632400e+04,NaN,0,0,0,0,0,0,0,0,0,0,0,9
5,0001634117,0001634117-26-000070,2026-07-09,2026-05-02,2026-07-09 22:15:49+02:00,10-K,1,1,bned-20260502.htm,https://www.sec.gov/Archives/edgar/data/1634117/000163411726000070/bned-20260502.htm,7.398970e+08,4.454580e+08,2.944390e+08,4.844630e+08,2.836040e+08,8418000.0,1.564365e+09,16872000.0,36538000.0,5.005700e+07,NaN,0,0,0,0,0,0,0,0,0,0,0,10
6,0000015847,0000015847-26-000020,2026-07-08,2026-04-30,2026-07-08 01:40:33+02:00,10-K,1,1,buks-20260430.htm,https://www.sec.gov/Archives/edgar/data/15847/000001584726000020/buks-20260430.htm,1.418420e+08,5.986400e+07,8.197800e+07,6.729700e+07,3.141900e+07,35124000.0,NaN,21933000.0,28453000.0,2.577900e+07,1933000.0,0,0,0,0,0,0,0,0,0,0,0,10
7,0001854183,0001683168-26-005342,2026-07-08,2026-04-30,2026-07-08 14:09:12+02:00,10-K,1,1,orion_i10k-043026.htm,https://www.sec.gov/Archives/edgar/data/1854183/000168316826005342/orion_i10k-043026.htm,3.702500e+04,NaN,-1.448220e+05,5.041000e+03,1.818480e+05,0.0,1.200000e+04,-40926.0,-40926.0,-3.182600e+04,NaN,0,0,0,0,0,0,0,0,0,0,0,9
8,0001848275,0001213900-26-075954,2026-07-07,2026-03-31,2026-07-07 22:15:58+02:00,10-K,1,1,ea0297046-10k_top.htm,https://www.sec.gov/Archives/edgar/data/1848275/000121390026075954/ea0297046-10k_top.htm,8.605694e+07,5.181180e+07,3.424514e+07,NaN,NaN,12989922.0,4.723765e+06,-1170823.0,-1177320.0,1.402180e+07,NaN,0,0,0,0,0,0,0,0,0,0,0,8
9,0001709542,0001683168-26-005287,2026-07-06,2025-12-31,2026-07-06 14:35:47+02:00,10-K,1,1,electronic_10k-123125.htm,https://www.sec.gov/Archives/edgar/data/1709542/000168316826005287/electronic_10k-123125.htm,1.189670e+05,NaN,-2.764362e+06,NaN,2.883329e+06,118967.0,0.000000e+00,-593833.0,-668829.0,6.441000e+04,NaN,0,0,0,0,0,0,0,0,0,0,0,8


In [16]:
con.sql("""
SELECT
    YEAR(filing_date) AS filing_year,
    COUNT(*) AS filings,
    COUNT(DISTINCT cik) AS filers,
    AVG(mapped_variables_available) AS mean_mapped_variables,
    SUM(CASE WHEN mapped_variables_available = 0 THEN 1 ELSE 0 END) AS zero_mapped_filings
FROM filing_financials_wide
GROUP BY filing_year
ORDER BY filing_year
""").df()

,filing_year,filings,filers,mean_mapped_variables,zero_mapped_filings
0,1993,4,4,0.000000,4.0
1,1994,1912,1900,0.000000,1912.0
2,1995,2218,2210,0.000000,2218.0
3,1996,4315,4256,0.000000,4315.0
4,1997,6698,6589,0.000000,6698.0
5,1998,6930,6804,0.000000,6930.0
6,1999,6761,6642,0.000000,6761.0
7,2000,6652,6517,0.000000,6652.0
8,2001,6248,6180,0.000000,6248.0
9,2002,6759,6612,0.000000,6759.0


## Before finalising the mapping

Inspect:

- concept coverage and fallback usage;
- duration distributions;
- ambiguous values;
- missingness by year and industry;
- whether additional concepts are justified;
- whether amendments should be handled separately.

Once stable, keep `config/financial_concept_map.csv` under version control.